# LLM Instance & Skill API Integration Test

This notebook demonstrates how an LLM instance can call the document-link-factory skill through an API.
Each request and response is printed without function encapsulation.

In [11]:
import sys
import json
import os
from pathlib import Path
from datetime import datetime
from typing import Dict, Any
import uuid

# For Jupyter notebook compatibility
project_root = Path('/Users/gamer/Documents/daily-builder/doc2skill/project')
sys.path.insert(0, str(project_root / 'src'))
os.chdir(project_root)

from novae_skill.runtime import dispatch_action

print('✓ Libraries imported successfully')
print(f'  Project root: {project_root}')
print(f'  Timestamp: {datetime.now()}')

✓ Libraries imported successfully
  Project root: /Users/gamer/Documents/daily-builder/doc2skill/project
  Timestamp: 2026-04-24 17:47:27.737112


## Skill API Server & LLM Instance Setup


**模拟一个"LLM（如Claude、GPT）通过API调用Skill包"的完整流程**

就像：
```
LLM说："帮我分析这些文档"
  ↓
通过API发送请求
  ↓
Skill包处理
  ↓
返回结果给LLM
  ↓
LLM继续工作
```

---

## 📝 各个单元格详解

### **单元格1-2：初始化**
```python
# 导入库
from novae_skill.runtime import dispatch_action
```
- **作用**：准备环境，导入Skill执行引擎
- **理解**：就像打开一个工具箱

---

### **单元格3-4：创建两个核心类**

#### **SkillAPIServer 类**
```python
class SkillAPIServer:
    def process_skill_request(self, request_body):
        # 处理请求 → 调用Skill → 返回结果
```

**理解**：
- 就像一个**餐厅的服务员**
- 顾客（LLM）提交订单（请求）
- 服务员把订单交给厨房（Skill）
- 厨房做好了返回给服务员
- 服务员再给顾客（返回响应）

```
LLM请求 → [服务员] → Skill处理 → [服务员] → LLM收到结果
```

#### **LLMInstance 类**
```python
class LLMInstance:
    def call_skill_api(self, skill_request):
        # 调用API
        response = self.api_server.process_skill_request(skill_request)
```

**理解**：
- 就像一个**LLM（如Claude）**
- 有一个方法 `call_skill_api()` 来调用Skill
- 跟踪所有的API调用历史

```
LLM ──(call_skill_api)──> SkillAPIServer ──(dispatch_action)──> Skill
                                ↓
                          记录调用日志
```

---

### **单元格5-6：TEST 1 - 发现文档页面**

这是一个**模拟的API对话**：

```python
# LLM构造请求
discover_request = {
    'action': 'discover_document_pages',
    'payload': {
        'entry_url': 'https://docs.python.org/3/',
        'candidate_links': [...]  # 候选页面列表
    }
}

# LLM通过API调用Skill
discover_response = llm.call_skill_api(discover_request)
```

**这段代码在做什么**：

```
LLM心想："我想发现Python文档中的所有教程页面"
         ↓
LLM构造请求：
{
  action: "discover_document_pages",
  payload: {
    entry_url: "https://docs.python.org/3/",
    candidate_links: [3个候选页面]
  }
}
         ↓
LLM调用：llm.call_skill_api(discover_request)
         ↓
SkillAPIServer收到请求
         ↓
Skill执行 dispatch_action('discover_document_pages', payload)
         ↓
返回结果给LLM：
{
  request_id: "abc123",
  status: "success",
  result: {
    page_count: 8,
    discovered_pages: [所有发现的页面]
  }
}
         ↓
LLM打印结果
```

**输出示例**：
```
📤 LLM → API Request:
  Action: discover_document_pages
  Entry URL: https://docs.python.org/3/
  Candidate Links: 3

📥 API → LLM Response:
  Request ID: abc12345
  Status: success
  Pages Discovered: 8
  - Getting Started (tutorial)
  - Library Docs (reference)
  - ...
```

---

### **单元格7-8：TEST 2-5 - 其他4个操作**

这个单元格一次性执行了4个API调用：

#### **TEST 2：分类（Classify）**
```python
classify_request = {
    'action': 'classify_document_pages',
    'payload': {
        'pages': discover_response['result']['discovered_pages'],
        # 告诉Skill：保留教程类，排除参考类
        'keep_kinds': ['tutorial', 'guide'],
        'exclude_kinds': ['reference', 'blog']
    }
}
```

**目的**：从8个发现的页面中，过滤出有用的（教程）

#### **TEST 3：提取能力（Extract）**
```python
extract_request = {
    'action': 'extract_document_capabilities',
    'payload': {
        'pages': classify_resp['result']['included_pages'],
        # 这些页面能教什么
        'page_evidence': [
            {'name': 'python_setup', 'purpose': 'Install Python'},
            {'name': 'string_fmt', 'purpose': 'Format strings'}
        ]
    }
}
```

**目的**：从页面中提取出来Skill能教的功能

#### **TEST 4：规范化（Normalize）**
```python
normalize_req = {
    'action': 'normalize_capabilities',
    'payload': {
        'capabilities': extract_resp['result']['capabilities'],
        'merge_strategy': 'collapse_duplicates'  # 合并重复的
    }
}
```

**目的**：整理和去重提取出的功能

#### **TEST 5：设计包（Design）**
```python
design_req = {
    'action': 'design_skill_package',
    'payload': {
        'project_name': 'python-core-skill',
        'skill_name': 'python-essentials',
        'normalized_pages': ...,
        'capability_model': ...  # 设计最终的Skill包
    }
}
```

**目的**：根据以上信息，设计最终的Skill包结构

---

### **单元格9-10：TEST 6-7 - 生成和验证**

#### **TEST 6：生成文件（Generate）**
```python
generate_req = {
    'action': 'generate_skill_files',
    'payload': {'package_plan': plan}
}
generate_resp = llm.call_skill_api(generate_req)

print(f'Generated Files: {len(files)}')
```

**目的**：根据设计，生成实际的skill文件和代码

#### **TEST 7：验证包（Validate）**
```python
validate_req = {
    'action': 'validate_skill_package',
    'payload': {'package_plan': plan, 'files': files}
}
validate_resp = llm.call_skill_api(validate_req)

print(f'Validation: {"✓ VALID" if result["valid"] else "✗ INVALID"}')
```

**目的**：验证生成的Skill包是否完整、正确

---

## 🔄 完整流程可视化

```
LLM系统                      Skill API Server           Skill包
   │                              │                       │
   ├──(1) discover request────────→ API ──→ dispatch_action
   │                              │            (discover)
   ←───(1) discover response──────┤                       │
   │                              │                       │
   ├──(2) classify request────────→ API ──→ dispatch_action
   │                              │            (classify)
   ←───(2) classify response──────┤                       │
   │                              │                       │
   ├──(3) extract request────────→ API ──→ dispatch_action
   │                              │            (extract)
   ←───(3) extract response──────┤                       │
   │                              │                       │
   ├──(4) normalize request──────→ API ──→ dispatch_action
   │                              │            (normalize)
   ←───(4) normalize response────┤                       │
   │                              │                       │
   ├──(5) design request────────→ API ──→ dispatch_action
   │                              │            (design)
   ←───(5) design response───────┤                       │
   │                              │                       │
   ├──(6) generate request──────→ API ──→ dispatch_action
   │                              │            (generate)
   ←───(6) generate response────┤                       │
   │                              │                       │
   ├──(7) validate request──────→ API ──→ dispatch_action
   │                              │            (validate)
   ←───(7) validate response────┤                       │
   │                              │                       │
   最终得到一个     Skill包生成完成！
   完整的Skill包
```

---

## 🎯 这个Notebook的意义

**在现实中**：
- 这演示了一个LLM（如Claude、GPT在你的代码中使用）如何**通过API接口**调用Skill包
- 展示了**完整的工作流程**：从发现→分类→提取→设计→生成→验证
- 记录了所有的**请求-响应对话**

**为什么要模拟这个过程**？
- ✅ 验证API接口是否正常工作
- ✅ 展示LLM和Skill包如何交互
- ✅ 测试完整的workflow是否能正确执行
- ✅ 作为其他系统集成时的参考

---

## 💡 简单类比

这就像：

```
你：我想从GitHub上的教程生成一个编程教学工具

系统做的7步：
1️⃣ 发现：从GitHub发现有哪些教程页面
2️⃣ 分类：选出好的教程，排除垃圾内容
3️⃣ 提取：从选中的教程中提取核心知识点
4️⃣ 规范化：整理这些知识点，去重合并
5️⃣ 设计：规划这个教学工具的结构
6️⃣ 生成：生成实际的代码文件
7️⃣ 验证：检查生成的工具是否完整

最后：✓ 你得到一个可用的教学工具Skill包
```



In [12]:
class SkillAPIServer:
    def __init__(self):
        self.request_log = []
        self.response_log = []
    
    def process_skill_request(self, request_body: Dict[str, Any]) -> Dict[str, Any]:
        request_id = str(uuid.uuid4())[:8]
        timestamp = datetime.now().isoformat()
        
        action_name = request_body.get('action')
        payload = request_body.get('payload', {})
        
        self.request_log.append({'request_id': request_id, 'action': action_name, 'timestamp': timestamp})
        
        try:
            result = dispatch_action(action_name, payload)
            status = 'success'
            error = None
        except Exception as e:
            status = 'error'
            error = str(e)
            result = {}
        
        response = {
            'request_id': request_id,
            'status': status,
            'action': action_name,
            'timestamp': timestamp,
            'result': result if status == 'success' else None,
            'error': error if status == 'error' else None
        }
        
        self.response_log.append({'request_id': request_id, 'status': status})
        return response

class LLMInstance:
    def __init__(self, api_server: SkillAPIServer, model_name: str = 'gpt-4-like'):
        self.api_server = api_server
        self.model_name = model_name
        self.conversation_history = []
        self.skill_calls = []
    
    def call_skill_api(self, skill_request: Dict[str, Any]) -> Dict[str, Any]:
        response = self.api_server.process_skill_request(skill_request)
        self.skill_calls.append({
            'timestamp': datetime.now().isoformat(),
            'action': skill_request.get('action'),
            'status': response['status'],
            'request_id': response['request_id']
        })
        return response
    
    def add_to_history(self, message: str):
        self.conversation_history.append({'message': message, 'timestamp': datetime.now().isoformat()})

# Initialize
api_server = SkillAPIServer()
llm = LLMInstance(api_server, model_name='deepseek-v4-flash')

print('✓ Skill API Server initialized')
print('✓ LLM Instance initialized')

✓ Skill API Server initialized
✓ LLM Instance initialized


## Test 1: LLM Discovers Pages via API

In [9]:
print('\n' + '=' * 70)
print('TEST 1: LLM CALLS DISCOVER_DOCUMENT_PAGES')
print('=' * 70)

discover_request = {
    'action': 'discover_document_pages',
    'payload': {
        'entry_url': 'https://docs.python.org/3/',
        'candidate_links': [
            {'title': 'Getting Started', 'url': 'https://docs.python.org/3/tutorial/', 'kind': 'tutorial', 'priority': 'high', 'evidence': ['sidebar'], 'reason': 'official'},
            {'title': 'Library Docs', 'url': 'https://docs.python.org/3/library/', 'kind': 'reference', 'priority': 'medium', 'evidence': ['breadcrumb'], 'reason': 'api'},
            {'title': 'FAQ', 'url': 'https://docs.python.org/3/faq/', 'kind': 'guide', 'priority': 'medium', 'evidence': ['nav'], 'reason': 'faq'}
        ],
        'navigation_hints': ['tutorial', 'guide', 'quickstart'],
        'exclusion_hints': ['blog', 'announce'],
    }
}

print('\n📤 LLM → API Request:')
print(f'  Action: {discover_request["action"]}')
print(f'  Entry URL: {discover_request["payload"]["entry_url"]}')
print(f'  Candidate Links: {len(discover_request["payload"]["candidate_links"])}')

discover_response = llm.call_skill_api(discover_request)

print('\n📥 API → LLM Response:')
print(f'  Request ID: {discover_response["request_id"]}')
print(f'  Status: {discover_response["status"]}')
print(f'  Pages Discovered: {discover_response["result"]["page_count"]}')
print('\n  Discovered Pages:')
for page in discover_response['result']['discovered_pages']:
    print(f'    - {page["title"]} ({page["kind"]})')


TEST 1: LLM CALLS DISCOVER_DOCUMENT_PAGES

📤 LLM → API Request:
  Action: discover_document_pages
  Entry URL: https://docs.python.org/3/
  Candidate Links: 3

📥 API → LLM Response:
  Request ID: 6b81a97e
  Status: success
  Pages Discovered: 3

  Discovered Pages:
    - Getting Started (tutorial)
    - Library Docs (reference)
    - FAQ (guide)


## Test 2-5: Additional API Calls

Continue with classify, extract, normalize, and design actions.

In [10]:
# Classify
classify_request = {
    'action': 'classify_document_pages',
    'payload': {
        'pages': discover_response['result']['discovered_pages'],
        'keep_kinds': ['tutorial', 'guide', 'example', 'quickstart'],
        'exclude_kinds': ['reference', 'changelog', 'blog'],
    }
}

print('\n' + '=' * 70)
print('TEST 2: CLASSIFY_DOCUMENT_PAGES')
print('=' * 70)

classify_resp = llm.call_skill_api(classify_request)
print(f'\n✓ Kept Pages: {classify_resp["result"]["kept_count"]}')
print(f'✗ Excluded Pages: {classify_resp["result"]["excluded_count"]}')

# Extract
page_evidence = [
    {'name': 'python_setup', 'purpose': 'Install Python', 'inputs': ['os'], 'outputs': ['path'], 'dependencies': ['pm'], 'risk_points': ['PATH'], 'evidence_pages': ['https://docs.python.org/3/tutorial/'], 'notes': []},
    {'name': 'string_fmt', 'purpose': 'Format strings', 'inputs': ['template'], 'outputs': ['result'], 'dependencies': [], 'risk_points': ['injection'], 'evidence_pages': ['https://docs.python.org/3/tutorial/'], 'notes': []}
]

extract_request = {'action': 'extract_document_capabilities', 'payload': {'pages': classify_resp['result']['included_pages'], 'page_evidence': page_evidence, 'target_language': 'python', 'constraints': {}}}
extract_resp = llm.call_skill_api(extract_request)

print('\n' + '=' * 70)
print('TEST 3: EXTRACT_DOCUMENT_CAPABILITIES')
print('=' * 70)
print(f'\nCapabilities Extracted: {len(extract_resp["result"]["capabilities"])}')

# Normalize
normalize_req = {'action': 'normalize_capabilities', 'payload': {'capabilities': extract_resp['result']['capabilities'], 'merge_strategy': 'collapse_duplicates', 'dedupe_keys': ['name']}}
normalize_resp = llm.call_skill_api(normalize_req)

print('\n' + '=' * 70)
print('TEST 4: NORMALIZE_CAPABILITIES')
print('=' * 70)
print(f'\nNormalized: {len(normalize_resp["result"]["normalized_capabilities"])}')

# Design
design_req = {'action': 'design_skill_package', 'payload': {'entry_url': 'https://docs.python.org/3/', 'project_name': 'python-core-skill', 'skill_name': 'python-essentials', 'constraints': {'target_language': 'python', 'target_skill_format': 'openapi', 'allow_third_party_libs': True, 'enable_local_cache': False, 'enable_vector_index': False, 'output_dir': './project'}, 'normalized_pages': classify_resp['result']['included_pages'], 'capability_model': normalize_resp['result']['normalized_capabilities']}}
design_resp = llm.call_skill_api(design_req)

print('\n' + '=' * 70)
print('TEST 5: DESIGN_SKILL_PACKAGE')
print('=' * 70)
plan = design_resp['result']['package_plan']
print(f'\nPackage: {plan["project_name"]}')
print(f'  Skill: {plan["skill_name"]}')
print(f'  Files: {len(plan["file_tree"])}')


TEST 2: CLASSIFY_DOCUMENT_PAGES

✓ Kept Pages: 2
✗ Excluded Pages: 1

TEST 3: EXTRACT_DOCUMENT_CAPABILITIES

Capabilities Extracted: 2

TEST 4: NORMALIZE_CAPABILITIES

Normalized: 2

TEST 5: DESIGN_SKILL_PACKAGE

Package: python-core-skill
  Skill: python-essentials
  Files: 14


## Test 6-7: Generate & Validate

Final generation and validation steps.

In [ ]:
# Generate Files
generate_req = {'action': 'generate_skill_files', 'payload': {'package_plan': plan}}
generate_resp = llm.call_skill_api(generate_req)

print('\n' + '=' * 70)
print('TEST 6: GENERATE_SKILL_FILES')
print('=' * 70)

files = generate_resp['result']['files']
print(f'\nGenerated Files: {len(files)}')
print(f'Total Size: {sum(len(c) for c in files.values())} bytes')

# Validate
validate_req = {'action': 'validate_skill_package', 'payload': {'package_plan': plan, 'files': files}}
validate_resp = llm.call_skill_api(validate_req)

print('\n' + '=' * 70)
print('TEST 7: VALIDATE_SKILL_PACKAGE')
print('=' * 70)

result = validate_resp['result']
print(f'\nValidation: {"✓ VALID" if result["valid"] else "✗ INVALID"}')
print(f'Missing Files: {len(result["missing_files"])}')
print(f'Warnings: {len(result["warnings"])}')

print(f'\n✓ All 7 API tests completed successfully!')
print(f'  Total LLM-API interactions: {len(llm.skill_calls)}')
print(f'  Total requests processed: {len(api_server.request_log)}')
print(f'  Total responses sent: {len(api_server.response_log)}')